# Machine Learning Assignment 2 — Task 2
## Heart Disease Risk Prediction System
---
**Student Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Degree Program:** BS Software Engineering (BSE)  
**Course:** Machine Learning (CSC-461)  

---
### Cloud Deployment & Repository Links
- **Live Cloud Deployment (Modal):** [https://mhassan0000--heart-disease-predictor-run.modal.run](https://mhassan0000--heart-disease-predictor-run.modal.run)
- **GitHub Repository:** [https://github.com/MHassan0000/ML-Prediction-Models](https://github.com/MHassan0000/ML-Prediction-Models)

---
### Assignment Constraint Adherence
Following the specific instructor requirements:
- **Dataset:** UCI Heart Disease Dataset (`heart.csv`)
- **Constraint:** Select **only 4 input attributes** from the dataset.
- **Workflow:** Execute the exact four phases of the ML cycle modeled in Task 1.

#### Selected 4 Input Attributes:
1. `age`: Patient Age in years
2. `thalach`: Maximum Heart Rate Achieved during stress test
3. `chol`: Serum Cholesterol in mg/dl
4. `cp`: Chest Pain Type (Categorical: 0 = Typical Angina, 1 = Atypical Angina, 2 = Non-anginal, 3 = Asymptomatic)



## Step 0: Environment Setup & Library Imports
We initialize the required environment and libraries.



In [ ]:
# Environment & Library Initialization
# Author: Muhammad Hassan Yousaf (FA24-BSE-082)

import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("[INFO] Libraries loaded successfully.")


## Step 1: Data Collection
We load the UCI Heart Disease dataset (`heart.csv`).



In [ ]:
# Data Collection Phase
data_candidates = ["data/heart.csv", "heart.csv"]
dataset_path = next((p for p in data_candidates if os.path.exists(p)), None)

if dataset_path:
    df_heart = pd.read_csv(dataset_path)
    print(f"[INFO] Loaded Heart Disease dataset from: {dataset_path}")
else:
    raise FileNotFoundError("heart.csv dataset not found in data/ or root directory.")

print(f"Dataset Dimensionality: {df_heart.shape[0]} rows x {df_heart.shape[1]} columns")
df_heart.head(8)


## Step 2: Data Understanding & Exploratory Data Analysis (EDA)
We examine the distribution of cardiovascular features and target correlation.



In [ ]:
# Dataset Structure & Data Types
print("--- Heart Disease Dataset Info ---")
df_heart.info()


In [ ]:
# Summary Statistics
df_heart.describe().T


In [ ]:
# Target Balance (0 = No Heart Disease, 1 = Heart Disease)
print(df_heart["target"].value_counts())


In [ ]:
# Exploratory Visualizations for Selected 4 Attributes
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Exploratory Data Analysis: Heart Disease Predictors (Hassan Yousaf)", fontsize=15, fontweight="bold")

# 1. Target Distribution
sns.countplot(data=df_heart, x="target", ax=axes[0, 0], palette="Reds")
axes[0, 0].set_title("Disease Class Balance (0 = Healthy, 1 = Disease)")
axes[0, 0].set_xlabel("Target Status")

# 2. Chest Pain Type vs Disease
sns.countplot(data=df_heart, x="cp", hue="target", ax=axes[0, 1], palette="coolwarm")
axes[0, 1].set_title("Heart Disease Incidence by Chest Pain Type")
axes[0, 1].set_xlabel("Chest Pain Type (cp)")

# 3. Maximum Heart Rate (thalach) vs Disease
sns.boxplot(data=df_heart, x="target", y="thalach", ax=axes[1, 0], palette="Set1")
axes[1, 0].set_title("Max Heart Rate (thalach) Distribution by Status")
axes[1, 0].set_xlabel("Target Status")

# 4. Serum Cholesterol (chol) vs Age Scatter
sns.scatterplot(data=df_heart, x="age", y="chol", hue="target", ax=axes[1, 1], palette="magma", alpha=0.8)
axes[1, 1].set_title("Age vs Cholesterol by Target Status")
axes[1, 1].set_xlabel("Age (years)")

plt.tight_layout()
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/heart_disease_eda.png", dpi=120)
plt.show()


## Step 3: Data Preprocessing & Constraint Adherence
We explicitly isolate the **4 mandated features**: `age`, `thalach`, `chol`, `cp` and the target `target`.



In [ ]:
# Feature Isolation (4 Input Attributes Constraint)
selected_features = ["age", "thalach", "chol", "cp"]
target_name = "target"

df_selected = df_heart[selected_features + [target_name]].dropna().copy()
print(f"Dataset Shape with 4 Selected Features: {df_selected.shape}")

# Fit LabelEncoder for 'cp' categorical encoding
le_cp = LabelEncoder()
df_selected["cp"] = le_cp.fit_transform(df_selected["cp"])

os.makedirs("data", exist_ok=True)
df_selected.to_csv("data/heart_disease_encoded.csv", index=False)
df_selected.head()


## Step 4: Model Training — Support Vector Classifier (SVC)
We partition the dataset into 80% training and 20% testing subsets (`random_state=42`) and fit the Support Vector Classifier with an RBF kernel.



In [ ]:
# Train / Test Splitting
X = df_selected[selected_features]
y = df_selected[target_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples:  {X_test.shape[0]}")

# SVC Training
svc_heart = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svc_heart.fit(X_train, y_train)
print("[SUCCESS] Heart Disease SVC Model trained successfully.")


## Step 5: Comprehensive Model Evaluation
We evaluate precision, recall, F1-score, and confusion matrix.



In [ ]:
# Model Performance Evaluation
y_pred = svc_heart.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"=== Model Test Accuracy: {acc * 100:.2f}% ===\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["Healthy (0)", "Disease (1)"]))

# Confusion Matrix Heatmap
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Predicted: Healthy", "Predicted: Disease"],
            yticklabels=["Actual: Healthy", "Actual: Disease"])
plt.title(f"Heart Disease SVC Confusion Matrix (Acc: {acc*100:.1f}%)", fontweight="bold")
plt.tight_layout()
plt.savefig("reports/heart_disease_confusion_matrix.png", dpi=120)
plt.show()


## Step 6: Clinical Inference Demonstration
We evaluate synthetic patient records across diverse clinical profiles.



In [ ]:
# Clinical Testing Profiles
patient_profiles = pd.DataFrame([
    {"age": 45, "thalach": 175, "chol": 190, "cp": 1},
    {"age": 62, "thalach": 120, "chol": 290, "cp": 0},
    {"age": 55, "thalach": 150, "chol": 240, "cp": 2},
    {"age": 67, "thalach": 105, "chol": 310, "cp": 0},
    {"age": 40, "thalach": 180, "chol": 170, "cp": 3}
])

preds = svc_heart.predict(patient_profiles[selected_features])
distances = svc_heart.decision_function(patient_profiles[selected_features])

patient_profiles["Prediction"] = ["Heart Disease" if p == 1 else "No Disease" for p in preds]
patient_profiles["Hyperplane Distance"] = np.round(distances, 3)

os.makedirs("reports", exist_ok=True)
patient_profiles.to_csv("reports/heart_disease_predictions.csv", index=False)
patient_profiles


## Step 7: Model Artifact Serialization & Verification
We persist the model and label encoder artifacts for cloud deployment.



In [ ]:
# Artifact Serialization
os.makedirs("models", exist_ok=True)

with open("models/heart_disease_svc_model.pkl", "wb") as f:
    pickle.dump(svc_heart, f)
with open("models/heart_disease_le_cp.pkl", "wb") as f:
    pickle.dump(le_cp, f)

# Root compatibility copies
with open("heart_disease_svc_model.pkl", "wb") as f:
    pickle.dump(svc_heart, f)
with open("heart_disease_le_cp.pkl", "wb") as f:
    pickle.dump(le_cp, f)

print("[SUCCESS] Heart disease model & encoder artifacts serialized successfully.")


## Summary & Conclusion
- **Student:** Muhammad Hassan Yousaf (FA24-BSE-082)
- **Model:** Support Vector Classifier (RBF Kernel)
- **Constraint:** Strictly 4 Input Attributes Selected (`age`, `thalach`, `chol`, `cp`)
- **Live Deployment Link:** [https://mhassan0000--heart-disease-predictor-run.modal.run](https://mhassan0000--heart-disease-predictor-run.modal.run)

